In [ ]:
#COLORMAP TEST

## TO DRAW CENTERLINE WITH OBJECTS! VERY IMPORTANT AND USEFUL

In [1]:
#import pckgs
import cv2
import tifffile as tiff
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from natsort import natsorted
import re
import csv
import seaborn as sns

In [2]:
from matplotlib import cm

In [3]:
import matplotlib.colors

In [4]:
from sklearn import preprocessing


In [5]:
#with objects, draw spline centerline
input_filename='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/btf_all/2020-07-01_13-21-00_chemotaxisl_worm1-channel-0-bigtiff.btf'
#'/groups/zimmer/shared_projects/Barbara_Ulises/Labeller_Test/multiple_skeleton/cropped_img_worm0.tiff'

        
csv_path='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/skeleton_after_head_and_tail_from_unet_all_long_job/2020-07-01_13-21-00_chemotaxisl_worm1-'

X_df=pd.read_csv(csv_path+'_spline_X_coords.csv', header=None)
Y_df=pd.read_csv(csv_path+'_spline_Y_coords.csv', header=None)
K_df=pd.read_csv(csv_path+'_spline_K.csv', header=None)


In [6]:
output_filename='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/skeleton_after_head_and_tail_from_unet_all_long_job_videos/2020-07-01_13-21-00_chemotaxisl_worm1_labeled_colormap_segment_good_inverted.btf'


In [ ]:
#Normalize your values with the preprocessing tools from sklearn

In [7]:
scaler = preprocessing.MinMaxScaler()
min_val=-0.03
max_val=0.03
x_sample = [min_val, max_val]
scaler.fit(np.array(x_sample)[:, np.newaxis]) # reshape data to satisfy fit() method requirements

K_df_norm=scaler.transform(K_df)

In [10]:
#loop
#K_df_norm=K_df_norm[105600:114000]
with tiff.TiffWriter(output_filename, bigtiff=True) as tif_writer:
    with tiff.TiffFile(input_filename, multifile=False) as tif:
        for i, page in enumerate(tif.pages):#[105600:114000]):
            if i <105600:continue #print(i)
            img=page.asarray()
            #convert to BRG (3 channels)
            img=cv2.cvtColor(img,cv2.COLOR_GRAY2BGR)
            #for every spline value
            for K_idx, K_value in enumerate(K_df_norm[i]):
                #if there are nans (centerline does not exist, continue)
                if np.isnan(X_df.iloc[i][K_idx])==True: continue
                x=int(X_df.iloc[i][K_idx])
                y=int(Y_df.iloc[i][K_idx])  
                
                #normalize k value to 255, important to do it. 0.03 is close to the max value
                K_color=cm.bwr(K_value)#, bytes=True)
                K_color=tuple([255*x for x in K_color]) #if bytes=False
                
                cv2.circle(img,(y,x), 3, K_color[:3],-1)
#                 plt.imshow(img)
#                 plt.show()
            tif_writer.write(img, contiguous=True)
            if i>114000: break
